# Titanic Survival Prediction with a Decision Tree

**Goal:** train an interpretable classifier that predicts whether a passenger survived using `Pclass`, `Sex`, `Age`, and `Fare`.

This notebook follows the same reproducible pipeline used by the command-line project. It expects Kaggle's labelled `train.csv` at `data/train.csv`. The dataset is not included in the repository; see [`data/README.md`](../data/README.md) for setup.

**Target:** `Survived` (`0` = did not survive; `1` = survived).


## 1. Imports and reproducibility

Pandas is used for loading and inspecting tabular data. scikit-learn handles the train/validation split, preprocessing, classifier, and metrics. A fixed random seed makes the split reproducible.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.tree import plot_tree

ROOT = Path.cwd()
if not (ROOT / "src" / "titanic_tree").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
from titanic_tree.model import FEATURES, TARGET, load_dataset, build_pipeline, original_feature_importance

DATA_PATH = ROOT / "data" / "train.csv"
RANDOM_STATE = 42
pd.set_option("display.max_columns", 20)


## 2. Load and inspect the dataset

The labelled `train.csv` file is used because the `Survived` column is required for training and evaluation.

In [2]:
df = load_dataset(DATA_PATH)
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")
display(df[FEATURES + [TARGET]].head())
print("\nData types and non-null counts:")
df.info()


Dataset shape: 891 rows x 12 columns


,Pclass,Age,Fare,Sex,Survived
0,3,22.0,7.2500,male,0
1,1,38.0,71.2833,female,1
2,3,26.0,7.9250,female,1
3,1,35.0,53.1000,female,1
4,3,35.0,8.0500,male,0



Data types and non-null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


## 3. Explore target balance and missing values

The training data is not perfectly balanced. Age is missing for some passengers, so a consistent imputation strategy is required. Missing values are handled inside the scikit-learn pipeline, which is fitted only on the training split.

In [3]:
print("Survival counts:")
display(df[TARGET].value_counts().rename(index={0: "Did not survive (0)", 1: "Survived (1)"}).to_frame("passengers"))
print("Missing values in selected features before preprocessing:")
display(df[FEATURES].isna().sum().rename("missing_count").to_frame())

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
counts = df[TARGET].value_counts().reindex([0, 1], fill_value=0)
axes[0].bar(["Did not survive", "Survived"], counts.values)
axes[0].set_title("Survival labels")
axes[0].set_ylabel("Passengers")
missing = df[FEATURES].isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]
if missing.empty:
    axes[1].text(0.5, 0.5, "No missing values", ha="center", va="center")
    axes[1].set_axis_off()
else:
    axes[1].bar(missing.index, missing.values)
    axes[1].set_title("Missing values before preprocessing")
    axes[1].set_ylabel("Missing entries")
fig.tight_layout()
plt.show()


Survival counts:


,passengers
Survived,
Did not survive (0),549
Survived (1),342


Missing values in selected features before preprocessing:


,missing_count
Pclass,0
Age,177
Fare,0
Sex,0


/tmp/ipykernel_1758/2424919283.py:21: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Select features and split the data

We use four inputs from the assignment. `Survived` is the label to predict. The validation set is held aside until after preprocessing and model fitting; stratification preserves the survival-class ratio in both splits.

In [4]:
X = df[FEATURES].copy()
y = df[TARGET].copy()
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
print(f"Training rows: {len(X_train)}")
print(f"Validation rows: {len(X_valid)}")
print(f"Input features: {FEATURES}")


Training rows: 712
Validation rows: 179
Input features: ['Pclass', 'Age', 'Fare', 'Sex']


## 5. Build and train the Decision Tree

The pipeline uses median imputation for numeric fields, most-frequent imputation for the categorical `Sex` field, and one-hot encoding for sex. A shallow tree (maximum depth 3) is chosen to keep the rules easy to inspect and explain.

In [5]:
model = build_pipeline(max_depth=3, min_samples_leaf=8)
model.fit(X_train, y_train)
predictions = model.predict(X_valid)
print("Model trained successfully.")


Model trained successfully.


## 6. Evaluate held-out predictions

Accuracy is the fraction of validation examples predicted correctly. Balanced accuracy gives equal weight to each class. Precision, recall, and F1 help explain performance for the passengers predicted to have survived.

In [6]:
majority_class = int(y_train.value_counts().idxmax())
metrics = {
    "Accuracy": accuracy_score(y_valid, predictions),
    "Balanced accuracy": balanced_accuracy_score(y_valid, predictions),
    "Precision (survived)": precision_score(y_valid, predictions, zero_division=0),
    "Recall (survived)": recall_score(y_valid, predictions, zero_division=0),
    "F1-score (survived)": f1_score(y_valid, predictions, zero_division=0),
    "Majority-class baseline accuracy": float((y_valid == majority_class).mean()),
}
display(pd.DataFrame({"metric": metrics.keys(), "value": [round(float(v), 3) for v in metrics.values()]}))
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay.from_predictions(
    y_valid, predictions, labels=[0, 1],
    display_labels=["Did not survive", "Survived"], cmap="Blues", ax=ax
)
ax.set_title("Validation confusion matrix")
fig.tight_layout()
plt.show()


,metric,value
0,Accuracy,0.793
1,Balanced accuracy,0.767
2,Precision (survived),0.776
3,Recall (survived),0.652
4,F1-score (survived),0.709
5,Majority-class baseline accuracy,0.615


/tmp/ipykernel_1758/135076384.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 7. Interpret feature importance

A decision tree's feature importance is based on the reductions in impurity produced by splits. The two encoded `Sex` indicator columns are added together so the graph reports the importance of the original input feature. Importance is model-specific and should not be interpreted as proof of causation.

In [7]:
importance = original_feature_importance(model)
display(importance.style.format({"importance": "{:.3f}"}))
plot_df = importance.sort_values("importance", ascending=True)
fig, ax = plt.subplots(figsize=(7.5, 4.3))
ax.barh(plot_df["feature"], plot_df["importance"])
ax.set_title("Decision Tree feature importance")
ax.set_xlabel("Aggregated importance")
for index, value in enumerate(plot_df["importance"]):
    ax.text(value + 0.008, index, f"{value:.3f}", va="center")
fig.tight_layout()
plt.show()


,feature,importance
0,Sex,0.644
1,Pclass,0.219
2,Age,0.091
3,Fare,0.046


/tmp/ipykernel_1758/236631985.py:11: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Inspect the learned tree

Each internal node is a threshold question. The path from the root to a leaf gives the class predicted for passengers satisfying those conditions.

In [8]:
preprocessor = model.named_steps["preprocessor"]
tree_features = [name.split("__", 1)[-1].replace("Sex_", "Sex = ") for name in preprocessor.get_feature_names_out()]
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(
    model.named_steps["classifier"],
    feature_names=tree_features,
    class_names=["Did not survive", "Survived"],
    filled=True, rounded=True, impurity=False, proportion=True,
    precision=2, fontsize=8, ax=ax
)
ax.set_title("Interpretable Decision Tree")
fig.tight_layout()
plt.show()


/tmp/ipykernel_1758/3947163007.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 9. Findings and limitations

In the included run on the standard 891-row training table, the decision tree achieved about **79.3% validation accuracy**, compared with a **61.5% majority-class baseline**. The most important input was **Sex** (about 64.4% aggregated importance), followed by `Pclass` (about 21.9%), `Age` (about 9.1%), and `Fare` (about 4.6%).

The validation confusion matrix contains 97 true negatives, 13 false positives, 24 false negatives, and 45 true positives. The model still misses some passengers who survived, so accuracy should not be viewed as the whole story.

These values correspond to a fixed random split and the included tree settings. Re-running with a different CSV, split, or hyperparameters can change the scores. This is a small educational dataset; feature importance describes patterns learned by the model and does not imply that any feature caused the outcome.